In [ ]:
!pip install faster-whisper yt-dlp
!apt-get install -y ffmpeg

In [ ]:
import os
import subprocess
from faster_whisper import WhisperModel
import torch

OUT_DIR = "output"
TEMP_DIR = "temp_audio"
LINKS_FILE = "links.txt"
COOKIES_FILE = "cookies.txt"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = WhisperModel(
    "large-v3",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)


In [ ]:
def download_audio(url, temp_dir=TEMP_DIR):
    cmd = [
        "yt-dlp",
        "--cookies", COOKIES_FILE,
        "--extract-audio",
        "--audio-format", "mp3",
        "--geo-bypass",
        "-o", os.path.join(temp_dir, "%(id)s.%(ext)s"),
        url
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"yt-dlp error:\n{result.stderr}")

    video_id = url.split("v=")[-1].split("&")[0]
    audio_path = os.path.join(temp_dir, f"{video_id}.mp3")

    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio not found: {video_id}")

    return audio_path, video_id

def transcribe_audio(audio_path):
    segments, _ = model.transcribe(
        audio_path,
        language="uk",
        beam_size=5
    )

    return " ".join(seg.text for seg in segments)

def load_links(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def main():
    urls = load_links(LINKS_FILE)

    for idx, url in enumerate(urls, 1):
        try:
            print(f"[{idx}/{len(urls)}] Processing: {url}")

            audio_path, video_id = download_audio(url)
            transcript = transcribe_audio(audio_path)

            txt_path = os.path.join(OUT_DIR, f"{video_id}.txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(transcript)

            os.remove(audio_path)

            print(f"Saved: {txt_path}\n")

        except Exception as e:
            print(f"Error: {url}\n{e}\n")

if __name__ == "__main__":
    main()